In [16]:
from qualtran import Bloq, CompositeBloq, BloqBuilder, Signature, Register
from qualtran import QBit, QInt, QUInt, QAny
from qualtran.drawing import show_bloq, show_call_graph, show_counts_sigma

from qualtran.bloqs.data_loading.qroam_clean import QROAMClean, QROAMCleanAdjoint
from qualtran.bloqs.state_preparation import StatePreparationViaRotations
from qualtran.bloqs.rotations import AddIntoPhaseGrad

from typing import *
import numpy as np
import sympy
import cirq
import tqdm

from qualtran import Bloq
from qualtran.symbolics import SymbolicInt
from qualtran.bloqs import block_encoding
from qualtran.resource_counting import get_bloq_call_graph
import attrs
from qualtran.bloqs.bookkeeping import Partition, Split, Join, Allocate, Free
from qualtran.bloqs.basic_gates import CSwap, TGate
from qualtran.drawing import show_call_graph
from qualtran.resource_counting import QECGatesCost, get_cost_value
from qualtran.resource_counting.generalizers import generalize_cswap_approx

# Test state preparation via rotations

In [13]:
# Example: dense/resource-only state preparation and unitary synthesis via QROAM rotations
from src.integrations.qualtran.state_prep_QROAM import StatePreparationViaQROAMRotations
from src.integrations.qualtran.unitary_synthesis_QROAM import UnitarySynthesisQROAM
from src.integrations.qualtran.utils import (
    get_Toffoli_counts,
    get_qubit_counts,
    get_ancilla_counts,
)

N = 128
phase_bitsize = 32
n_reflections = 128

for log_block_sizes in [0, 1, 2, 3, 4, 5]:
    print(
        f"--------------------------N={N}, b={phase_bitsize}, "
        f"K={n_reflections}, log_block_sizes={log_block_sizes}--------------------------"
    )

    sp_qroam = StatePreparationViaQROAMRotations.from_bitsize(
        n_coeff=N,
        phase_bitsize=phase_bitsize,
        log_block_sizes=log_block_sizes,
        adjoint_log_block_sizes=log_block_sizes,
    )

    unitary_synth_qroam = UnitarySynthesisQROAM.from_shape(
        (N, n_reflections),
        phase_bitsize=phase_bitsize,
        log_block_sizes=log_block_sizes,
        adjoint_log_block_sizes=log_block_sizes,
    )

    for name, bloq in [
        ("state preparation", sp_qroam),
        ("unitary synthesis", unitary_synth_qroam),
    ]:
        print(f"{name}:")
        print("  signature:", bloq.signature)
        print("  Toffoli count:", get_Toffoli_counts(bloq))
        print("  peak logical qubits:", get_qubit_counts(bloq))
        print("  intermediate ancillas:", get_ancilla_counts(bloq))


--------------------------N=128, b=32, K=128, log_block_sizes=0--------------------------
state preparation:
  signature: Signature((Register('target_state', dtype=QAny(bitsize=7), shape=(), side=Side.THRU), Register('phase_gradient', dtype=QAny(bitsize=32), shape=(), side=Side.THRU)))
  Toffoli count: 727
  peak logical qubits: 79
  intermediate ancillas: 40
unitary synthesis:
  signature: Signature((Register('reflection_ancilla', dtype=QBit(), shape=(), side=Side.THRU), Register('system', dtype=QAny(bitsize=7), shape=(), side=Side.THRU), Register('phase_gradient', dtype=QAny(bitsize=32), shape=(), side=Side.THRU)))
  Toffoli count: 190720
  peak logical qubits: 81
  intermediate ancillas: 41
--------------------------N=128, b=32, K=128, log_block_sizes=1--------------------------
state preparation:
  signature: Signature((Register('target_state', dtype=QAny(bitsize=7), shape=(), side=Side.THRU), Register('phase_gradient', dtype=QAny(bitsize=32), shape=(), side=Side.THRU)))
  Toffoli 

# Unitary synthesis QROAM Toffoli scaling

Compare dense, data-free unitary synthesis with `phase_bitsize = 32` and `n_reflections = N` for fixed `log_block_sizes = 0` versus the best scanned integer `log_block_sizes`.


In [17]:
# Toffoli scaling for dense unitary synthesis with K = N reflections
import matplotlib.pyplot as plt
import tqdm

from src.integrations.qualtran.unitary_synthesis_QROAM import UnitarySynthesisQROAM
from src.integrations.qualtran.utils import get_Toffoli_counts

phase_bitsize = 20
exponents = range(5, 20)
N_values = [2**exp for exp in exponents]

log0_counts = []
optimal_counts = []
optimal_log_block_sizes = []

for N in tqdm.tqdm(N_values):
    log_n = (N - 1).bit_length()

    log0_bloq = UnitarySynthesisQROAM.from_shape(
        (N, N),
        phase_bitsize=phase_bitsize,
        log_block_sizes=0,
        adjoint_log_block_sizes=0,
    )
    log0_count = int(get_Toffoli_counts(log0_bloq))

    candidates = []
    for log_block_sizes in range(log_n + 1):
        bloq = UnitarySynthesisQROAM.from_shape(
            (N, N),
            phase_bitsize=phase_bitsize,
            log_block_sizes=log_block_sizes,
            adjoint_log_block_sizes=log_block_sizes,
        )
        candidates.append((log_block_sizes, int(get_Toffoli_counts(bloq))))

    best_log_block_sizes, best_count = min(candidates, key=lambda item: item[1])
    log0_counts.append(log0_count)
    optimal_counts.append(best_count)
    optimal_log_block_sizes.append(best_log_block_sizes)

print("N	log_block_sizes=0	optimal_log_block_sizes	optimal")
for N, count0, best_log, best_count in zip(
    N_values, log0_counts, optimal_log_block_sizes, optimal_counts
):
    print(f"{N}	{count0}	{best_log}	{best_count}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(N_values, log0_counts, "o-", label="log_block_sizes = 0")
ax.loglog(N_values, optimal_counts, "s-", label="best scanned log_block_sizes")
ax.set_xlabel("N = n_reflections")
ax.set_ylabel("Toffoli count")
ax.set_title(f"QROAM unitary synthesis Toffoli counts, b = {phase_bitsize}")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()


 67%|██████▋   | 10/15 [02:22<02:53, 34.68s/it]

: 

: 

: 